# Load a Bloch project (`.blochproj`)

A **Bloch project** is the complete, re-openable GUI state: the sequence, the
phantom, the B1 fields, all GUI controls *and* the simulation results. It is
written by **File → Save Project**, by **Export selected…** in the
Simulations list and by parameter loops with *Results: Save to folder*. Open it
again with **File → Open Project**, or read it in Python as shown here.

This notebook shows how to

1. read the simulation parameters (flip angle, matrix, FOV, TR/TE, nucleus,
   B0, phantom, B1),
2. see which simulations a project holds and access each one, down to a
   single spin, voxel, pool or coil,
3. collect the runs of a parameter loop (one project per run).

If you only need the numbers of one result (signal, k-space, images) in a
format that does not need Bloch Simulator, use **Export Results** instead, see
[load_exported_results.ipynb](load_exported_results.ipynb).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from blochsimulator.project_io import load_project

# Change this to your project file.
PROJECT_FILE = Path("simulation.blochproj")

project = load_project(PROJECT_FILE)
sorted(project)

## What is in a project

`load_project` returns a dictionary:

| key | content |
| --- | --- |
| `legacy_result` | the **Free Mode** simulation (dict of arrays) or `None` |
| `sequence_result` | the **Sequence mode** simulation: a `SequenceSimulationResult`, or a `SequenceProbeResult` for the spin probe, or `None` |
| `program` | the sequence loaded in the Sequence workspace (`SequenceProgram`) |
| `phantom` | the phantom (`None` for a spin probe or Free Mode only) |
| `tx_field`, `rx_field` | transmit / receive B1 fields (`B1Field`) or `None` |
| `state` | all GUI controls, e.g. the Free Mode pulse in `state["rf"]` |

So **one project can hold two simulations at once**: the last Free Mode run
and the last Sequence (or Spin Probe) run. The cell below lists them.

In [ ]:
def list_simulations(project):
    """Return the simulations stored in one project, keyed by a short name."""
    simulations = {}
    if project["legacy_result"] is not None:
        simulations["free_mode"] = project["legacy_result"]
    result = project["sequence_result"]
    if result is not None:
        # The spin probe result has probe positions, a sequence result does not.
        name = "spin_probe" if hasattr(result, "positions_m") else "sequence"
        simulations[name] = result
    return simulations


simulations = list_simulations(project)
print("Simulations in this project:", list(simulations))

## Simulation parameters

The sequence parameters of a Sequence-mode run (flip angle, matrix, FOV,
TR, TE, nucleus, B0, …) are stored **with the result** in
`result.metadata["sequence_definitions"]`. Use these rather than
`program.metadata["definitions"]`: the program is whatever sequence was
loaded when the project was saved, which can differ from the one that
produced the result if the sequence was changed after the run.

Units are SI: FOV in m, TR/TE in s, flip angle in degrees. The available keys
depend on the sequence type.

In [ ]:
result = project["sequence_result"]
program = project["program"]
if result is not None:
    definitions = result.metadata.get("sequence_definitions", {})
elif program is not None:
    definitions = program.metadata.get("definitions", {})
else:
    definitions = {}

print("Sequence:", definitions.get("Name"))
for key in ("FlipAngleDeg", "MatrixSize", "FOV", "TR", "TE", "Nucleus", "FieldStrengthT"):
    print(f"  {key}: {definitions.get(key)}")
print(f"{len(definitions)} parameters in total:", ", ".join(sorted(definitions)))

### Phantom and B1

The phantom is restored as the same object the GUI used (e.g.
`SpectralPhantom`, `MousePerfusionPhantom`), including its pools/metabolites,
relaxation times and maps. The B1 fields are `B1Field` objects with the
relative field map in `data`.

In [ ]:
phantom = project["phantom"]
if phantom is not None:
    print(f"Phantom: {phantom.name} ({type(phantom).__name__})")
    print(f"  grid {phantom.shape}, FOV {np.asarray(phantom.fov) * 1e3} mm")
    for pool in getattr(phantom, "species", None) or getattr(phantom, "pools", None) or ():
        print(f"  pool {pool.name}: T1 = {pool.t1:g} s, T2 = {pool.t2:g} s")

for key in ("tx_field", "rx_field"):
    field = project[key]
    if field is not None:
        print(f"B1 {field.kind}: {field.name}, map {field.data.shape}, "
              f"FOV {np.asarray(field.fov_m) * 1e3} mm")
    else:
        print(f"B1 {key}: none (homogeneous)")

The Free Mode controls (pulse, tissue, positions and frequencies) are part of
the saved GUI state:

In [ ]:
state = project["state"]
print("Free Mode pulse:", state.get("rf"))
print("Free Mode tissue:", state.get("tissue"))
print("Free Mode spins:", state.get("simulation"))

## Free Mode: access a single spin

Free Mode simulates every combination of the chosen positions and frequency
offsets as an independent spin. `mx`, `my`, `mz` and `signal` have the shape
`(time, position, frequency)`; `positions` is `(position, 3)` in m and
`frequencies` is in Hz. Pick one spin by its position and frequency index.

In [ ]:
free_mode = simulations.get("free_mode")
if free_mode is not None:
    positions_mm = np.asarray(free_mode["positions"]) * 1e3
    frequencies_hz = np.asarray(free_mode["frequencies"])
    print("mx shape (time, position, frequency):", free_mode["mx"].shape)

    position_index = 0
    frequency_index = int(np.argmin(np.abs(frequencies_hz)))  # spin closest to 0 Hz
    time_ms = np.asarray(free_mode["time"]) * 1e3
    fig, ax = plt.subplots(figsize=(7, 3))
    for name in ("mx", "my", "mz"):
        ax.plot(time_ms, free_mode[name][:, position_index, frequency_index], label=name)
    ax.set(xlabel="time (ms)", ylabel="M / M0",
           title=f"spin at {positions_mm[position_index]} mm, "
                 f"{frequencies_hz[frequency_index]:g} Hz")
    ax.legend()
    plt.show()

## Sequence mode: signal, voxels, pools and coils

A `SequenceSimulationResult` holds

* `signal`: the ADC signal, `(adc,)` or `(coil, adc)` for several receive coils,
* `adc_times_s`: the time of every ADC sample,
* `final_magnetization`: `(..., 3)` Mx/My/Mz on the phantom grid at the end,
* for multi-pool phantoms `pool_names`, `species_signal` (signal per pool)
  and `final_pool_magnetization`,
* `metadata`: parameters, phantom summary, parameter loop and timing.

Cartesian sequences can be reconstructed directly.

In [ ]:
sequence = simulations.get("sequence")
if sequence is not None:
    print("signal:", sequence.signal.shape, " final magnetization:", sequence.final_magnetization.shape)

    # One voxel of the phantom grid, here the centre.
    centre = tuple(size // 2 for size in sequence.final_magnetization.shape[:-1])
    print(f"M in voxel {centre}:", sequence.final_magnetization[centre])

    # One pool (metabolite) of a multi-pool phantom.
    for index, pool in enumerate(sequence.pool_names):
        print(f"pool {pool}: max |signal| = {np.max(np.abs(sequence.species_signal[index])):.4g}")

    # One receive coil.
    if sequence.signal.ndim == 2:
        coil_signal = sequence.signal[0]

    if sequence.cartesian_acquisition is not None:
        image = sequence.reconstruct_cartesian(sequence.cartesian_acquisition)
        plt.imshow(np.abs(image), cmap="gray")
        plt.title("Cartesian reconstruction")
        plt.colorbar()
        plt.show()

### Reconstruction methods: Cartesian FFT or non-Cartesian NUFFT

How the images of a Sequence run are reconstructed depends on the readout
trajectory:

- **Cartesian** (FLASH, EPI, bSSFP, CSI): the samples lie on the k-space
  grid. The image is the centred inverse FFT of the label-sorted k-space.
- **Spiral** (non-Cartesian): the spiral layout selects the method. By
  default it is the density-compensated **NUFFT** (`"nufft"`). The
  alternatives are the iterative least-squares CG-NUFFT (`"cg"`) and linear
  gridding followed by an FFT (`"linear"`, blurrier, the legacy reference).
- **3D radial** (non-Cartesian): density-compensated trilinear gridding. The
  analysis notebook written next to a `.nc` export also shows the NUFFT of the
  same spokes.

Do not place non-Cartesian samples into an FFT as they come. A **NUFFT**
(`blochsimulator.sequence.nufft.NUFFTOperator(k_cyc_per_m, matrix, fov_m)`)
first weights the densely sampled k-space centre down (density compensation).
It then interpolates the samples onto an oversampled grid with a
Kaiser–Bessel kernel. Its `sampling_gap` is the largest gap between samples in
units of 1/FOV. Above 0.5 the trajectory is undersampled and aliases outside
`nyquist_fov_m()`.

The cell below lists the stored reconstructions. For a spiral run it also
plots the trajectory of one frame and compares the three methods.

In [ ]:
from blochsimulator.sequence import infer_spiral_acquisition

if sequence is not None:
    dataset = sequence.to_xarray()  # all stored reconstructions, named dimensions
    for name in ("cartesian_image", "cartesian_3d_image", "spiral_image",
                 "radial_3d_image", "csi_spatial_fid"):
        if name in dataset:
            print(f"{name}: dims={dataset[name].dims} shape={dataset[name].shape}")

    layouts = ("spiral_acquisition", "cartesian_acquisition", "cartesian_acquisition_frames",
               "cartesian_acquisition_volumes", "spectroscopic_acquisition")
    spiral = sequence.spiral_acquisition
    if program is not None and not any(sequence.metadata.get(key) for key in layouts):
        try:  # older projects of undeclared spirals carry no layout
            spiral = infer_spiral_acquisition(program)
        except (KeyError, ValueError):
            spiral = None
    if spiral is not None:
        frame = 0
        trajectory = spiral.trajectory(sequence.adc_gradient_moment_cyc_per_m, frame)
        operator = spiral.nufft_operator(trajectory)
        print(f"Spiral, stored method: {spiral.reconstruction_label}")
        print(f"Largest k-space gap: {operator.sampling_gap:.2f}/FOV (Nyquist: <= 0.5)")
        if operator.sampling_gap > 0.5:
            alias_free = " x ".join(f"{fov * 1e3:.0f}" for fov in operator.nyquist_fov_m())
            print(f"Undersampled: expect aliasing outside about {alias_free} mm.")
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].plot(trajectory[:, 0], trajectory[:, 1], lw=0.6)
        axes[0].set(title="Trajectory", xlabel="kx (1/m)", ylabel="ky (1/m)", aspect="equal")
        for ax, method in zip(axes[1:], spiral.RECONSTRUCTION_METHODS):
            image = spiral.reconstruct(sequence, frame, method=method)
            if image.ndim == 3:  # root sum of squares over receive coils
                image = np.sqrt(np.sum(np.abs(image) ** 2, axis=0))
            ax.imshow(np.abs(image), origin="lower", cmap="gray")
            ax.set_title(spiral.RECONSTRUCTION_LABELS[method])
            ax.axis("off")
        plt.show()

## Spin probe

A spin-probe result (`SequenceProbeResult`) holds `time_s`, `positions_m`,
`frequency_offsets_hz` and `magnetization` with shape
`(time, position, frequency, 3)`.

In [ ]:
probe = simulations.get("spin_probe")
if probe is not None:
    print("magnetization (time, position, frequency, xyz):", probe.magnetization.shape)
    spin = probe.magnetization[:, 0, 0]  # first position, first frequency
    plt.plot(np.asarray(probe.time_s) * 1e3, np.hypot(spin[:, 0], spin[:, 1]), label="|Mxy|")
    plt.plot(np.asarray(probe.time_s) * 1e3, spin[:, 2], label="Mz")
    plt.xlabel("time (ms)")
    plt.legend()
    plt.show()

## Many simulations: parameter loops and batch exports

A project holds one Sequence run. A **parameter loop** with *Results: Save to
folder* (or **Export selected…** with several entries of the Simulations
list) writes **one `.blochproj` per run** into a folder. Each run of a loop
stores the varied parameter in `result.metadata["parameter_loop"]` (`loop_id`,
`parameter_label`, `value`, `unit`, `index`, …), so you can pick single runs
or stack all of them. Set `LOOP_ID` if the folder holds several loops.

In [ ]:
# Change this to the folder of the loop export.
LOOP_FOLDER = PROJECT_FILE.parent
LOOP_ID = None

runs = []
for path in sorted(LOOP_FOLDER.glob("*.blochproj")):
    run_result = load_project(path)["sequence_result"]
    loop = None if run_result is None else run_result.metadata.get("parameter_loop")
    if loop is None or (LOOP_ID is not None and loop["loop_id"] != LOOP_ID):
        continue
    runs.append((loop["index"], loop, run_result, path))
runs.sort(key=lambda run: run[0])

for index, loop, run_result, path in runs:
    print(f"run {index}: {loop['parameter_label']} = {loop['value']:g} {loop['unit']} -> {path.name}")

In [ ]:
# Example: signal magnitude of every run against the loop parameter.
if runs:
    values = np.array([loop["value"] for _, loop, _, _ in runs])
    signal = np.array([np.max(np.abs(run_result.signal)) for _, _, run_result, _ in runs])
    plt.plot(values, signal, "o-")
    plt.xlabel(f"{runs[0][1]['parameter_label']} ({runs[0][1]['unit']})")
    plt.ylabel("max |signal|")
    plt.show()